# Distribution VAE — Quickstart

This notebook walks through the basic workflow:
1. Train on synthetic data
2. Inspect reconstructions
3. Explore the latent space
4. Encode distributions to latent vectors

In [ ]:
import torch
from dist_vae.model import DistributionVAE
from dist_vae.data import SyntheticDistributionDataset
from dist_vae.train import Trainer

## 1. Create Synthetic Data

In [ ]:
# Create synthetic dataset of Gaussian mixture distributions
train_dataset = SyntheticDistributionDataset(n_distributions=1000, grid_size=256, seed=42)
val_dataset = SyntheticDistributionDataset(n_distributions=200, grid_size=256, seed=123)
print(f"Train: {len(train_dataset)} distributions, Val: {len(val_dataset)} distributions")

## 2. Create and Train Model

In [ ]:
# Create model
model = DistributionVAE(grid_size=256, latent_dim=32, hidden_dim=128, beta=0.01)
print(model)

In [ ]:
# Train
config = {
    'training': {'epochs': 50, 'batch_size': 64, 'lr': 1e-3, 'weight_decay': 1e-4,
                 'grad_clip': 1.0, 'val_fraction': 0.1, 'seed': 42},
    'model': {'beta': 0.01, 'beta_warmup_epochs': 5},
    'loss': {'cramer': 1.0, 'wasserstein1': 0.0, 'ks_smooth': 0.0, 'ks_temperature': 0.01},
    'logging': {'wandb': False, 'print_every': 10, 'checkpoint_dir': 'checkpoints/'},
}
trainer = Trainer(model, train_dataset, val_dataset, config)
history = trainer.train()

## 3. Inspect Reconstructions

In [ ]:
from dist_vae.eval import plot_reconstructions, evaluate_reconstruction

metrics = evaluate_reconstruction(model, val_dataset)
print(metrics)

plot_reconstructions(model, val_dataset, n_examples=9)

## 4. Explore Latent Space

In [ ]:
from dist_vae.eval import plot_latent_space, plot_interpolations

plot_latent_space(model, val_dataset, method='pca')
plot_interpolations(model, val_dataset, idx_pairs=[(0, 1), (2, 3)], n_steps=8)

## 5. Encode Distributions

In [ ]:
# Encode all distributions to latent vectors
model.eval()
with torch.no_grad():
    grid, _, _ = val_dataset[0]
    mu, logvar = model.encoder(grid.unsqueeze(0))
    print(f"Latent vector shape: {mu.shape}")
    print(f"Latent vector: {mu[0, :8]}...")  # first 8 dims